<a href="https://colab.research.google.com/github/joanby/python-ml-course/blob/master/notebooks/T5%20-%202%20-%20Logistic%20Regression%20-%20Implementación-Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clonamos el repositorio para obtener los dataSet

In [1]:
!git clone https://github.com/DavidArroyoTorres/python-ml-course/

Cloning into 'python-ml-course'...
remote: Enumerating objects: 17912, done.
remote: Counting objects: 100% (154/154), done.
remote: Compressing objects: 100% (148/148), done.
remote: Total 17912 (delta 100), reused 6 (delta 6), pack-reused 17758 (from 2)
Receiving objects: 100% (17912/17912), 531.92 MiB | 18.48 MiB/s, done.
Resolving deltas: 100% (440/440), done.
Updating files: 100% (16940/16940), done.


# Damos acceso a nuestro Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Test it
!ls '/content/drive/My Drive'

# Implementación el método de la máxima verosimilitud para la regresión logística

### Definir la función de entorno L(b)

In [54]:
from IPython.display import display, Math, Latex
display(Math(r'L(p;Y)=\prod_{i=1}^n p_i^{y_i}(1-p_i)^{1-y_i}, p_i=P(Y_i=1|X_{i,1}=x_{i,1},...,X_{i,k}=x_{i,k}) \text{ es  la función de verosimilitud de Y|X.}'))

<IPython.core.display.Math object>

In [55]:
def likelihood(y, pi):
    import numpy as np
    prod = 1
    prod_in = list(range(len(y)))
    for i in range(len(y)):
        prod_in[i] = np.where(y[i]==1, pi[i], 1-pi[i])
        prod = prod * prod_in[i]
    return prod

### Calcular las probabilidades para cada observación

In [56]:
display(Math(r'P_i = P(x_i) = \frac{1}{1+e^{-\beta_0-\sum_{j=1}^k\beta_j\cdot x_{ij}}} '))

<IPython.core.display.Math object>

In [57]:
def logitprobs(X,beta): #Luego se ve el sentido, X=Matriz de tamaño nx(k+1) de los datos predictivos y una columna de 1.
    import numpy as np   #Beta=vector de dimensión k+1.
    n_rows = np.shape(X)[0] #n=nº filas
    n_cols = np.shape(X)[1] #k+1=nº columnas
    p_i=list(range(n_rows)) #[0,..., n-1], con el mismo propósito se podría usar [0,...,0] con n ceros.
    expon=list(range(n_rows)) #[0,..., n-1] con el mismo propósito se podría usar [0,...,0] con n ceros.
    for i in range(n_rows): #[0,..., n-1]
        expon[i] = 0
        for j in range(n_cols): #[0,..., k]
            ex=X[i][j] * beta[j]
            expon[i] = ex + expon[i]
        with np.errstate(divide="ignore", invalid="ignore"):
            p_i[i]=1/(1+np.exp(-expon[i]))
    return p_i

### Calcular la matriz diagonal W

In [58]:
display(Math(r'W= diag(P_i \cdot (1-P_i))_{i=1}^n'))

<IPython.core.display.Math object>

In [59]:
def findW(p_i):
    import numpy as np
    n = len(p_i)
    W = np.zeros(n*n).reshape(n,n)
    for i in range(n):
        print(i)
        W[i,i]=p_i[i]*(1-p_i[i])
        W[i,i].astype(float)
    return W

### Obtener la solución de la función logística

In [60]:
display(Math(r"\beta_{n+1} = \beta_n -\frac{f(\beta_n)}{f'(\beta_n)}"))
display(Math(r"f(\beta) = X(Y-P)"))
display(Math(r"f'(\beta) = XWX^T"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [61]:
def logistics(X, Y, limit):
    import numpy as np
    from numpy import linalg
    nrow = np.shape(X)[0]
    bias = np.ones(nrow).reshape(nrow,1)
    X_new = np.append(X, bias, axis = 1)
    ncol = np.shape(X_new)[1]
    beta = np.zeros(ncol).reshape(ncol,1)
    root_dif = np.array(range(1,ncol+1)).reshape(ncol,1)
    iter_i = 10000
    while(iter_i>limit):
        print("Iter:i"+str(iter_i) + ", limit:" + str(limit))
        p_i = logitprobs(X_new, beta)
        print("P_i:"+str(p_i))
        W = findW(p_i)
        print("W:"+str(W))
        num = (np.transpose(np.matrix(X_new))*np.matrix(Y - np.transpose(p_i)).transpose())
        den = (np.matrix(np.transpose(X_new))*np.matrix(W)*np.matrix(X_new))
        root_dif = np.array(linalg.inv(den)*num)
        beta = beta + root_dif
        print("Beta: "+str(beta))
        iter_i = np.sum(root_dif*root_dif)
        ll = likelihood(Y, p_i)
    return beta

## Comprobación experimental

In [62]:
import numpy as np

In [85]:
X = np.array(range(10)).reshape(10,1)

In [86]:
X

array([[0],
       [1],
       [2],
       [3],
       [4],
       [5],
       [6],
       [7],
       [8],
       [9]])

In [87]:
Y = [0,0,0,0,1,0,1,0,1,1]

In [88]:
bias = np.ones(10).reshape(10,1)
X_new = np.append(X,bias,axis=1)

In [67]:
X_new

array([[0., 1.],
       [1., 1.],
       [2., 1.],
       [3., 1.],
       [4., 1.],
       [5., 1.],
       [6., 1.],
       [7., 1.],
       [8., 1.],
       [9., 1.]])

In [68]:
a = logistics(X,Y,0.00001)

Iter:i10000, limit:1e-05
P_i:[array([0.5]), array([0.5]), array([0.5]), array([0.5]), array([0.5]), array([0.5]), array([0.5]), array([0.5]), array([0.5]), array([0.5])]
0
1
2
3
4
5
6
7
8
9
W:[[0.25 0.   0.   0.   0.   0.   0.   0.   0.   0.  ]
 [0.   0.25 0.   0.   0.   0.   0.   0.   0.   0.  ]
 [0.   0.   0.25 0.   0.   0.   0.   0.   0.   0.  ]
 [0.   0.   0.   0.25 0.   0.   0.   0.   0.   0.  ]
 [0.   0.   0.   0.   0.25 0.   0.   0.   0.   0.  ]
 [0.   0.   0.   0.   0.   0.25 0.   0.   0.   0.  ]
 [0.   0.   0.   0.   0.   0.   0.25 0.   0.   0.  ]
 [0.   0.   0.   0.   0.   0.   0.   0.25 0.   0.  ]
 [0.   0.   0.   0.   0.   0.   0.   0.   0.25 0.  ]
 [0.   0.   0.   0.   0.   0.   0.   0.   0.   0.25]]
Beta: [[ 0.43636364]
 [-2.36363636]]
Iter:i5.777190082644626, limit:1e-05
P_i:[array([0.08598797]), array([0.12705276]), array([0.18378532]), array([0.2583532]), array([0.35019508]), array([0.45467026]), array([0.56329497]), array([0.66616913]), array([0.75533524]), array([0.8

/tmp/ipykernel_7615/1963481205.py:7: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  W[i,i]=p_i[i]*(1-p_i[i])


In [69]:
ll = likelihood(Y, logitprobs(X,a))

In [70]:
ll

array([1.32622426e-06])

In [71]:
Y = 0.66220827 * X -3.69557172

# Con el paquete statsmodel de python
(Cuidado de no tomar Y = 0.66220827 * X -3.69557172, tomar los datos del comienzo de Comprobación experimental)

In [89]:
import statsmodels.api as sm
import pandas as pd
from pandas import Timestamp

In [90]:
#Y = (Y - np.min(Y))/np.ptp(Y) (No entiendo por qué pone esto)
logit_model = sm.Logit(Y,X_new)

In [91]:
result = logit_model.fit()

Optimization terminated successfully.
         Current function value: 0.431012
         Iterations 6


In [93]:
print(result.summary())

                           Logit Regression Results                           
Dep. Variable:                      y   No. Observations:                   10
Model:                          Logit   Df Residuals:                        8
Method:                           MLE   Df Model:                            1
Date:                Wed, 06 May 2026   Pseudo R-squ.:                  0.3596
Time:                        18:59:48   Log-Likelihood:                -4.3101
converged:                       True   LL-Null:                       -6.7301
Covariance Type:            nonrobust   LLR p-value:                   0.02781
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
x1             0.6622      0.400      1.655      0.098      -0.122       1.446
const         -3.6956      2.289     -1.615      0.106      -8.182       0.791


In [94]:
print(result.summary2())

                        Results: Logit
Model:              Logit            Method:           MLE     
Dependent Variable: y                Pseudo R-squared: 0.360   
Date:               2026-05-06 18:59 AIC:              12.6202 
No. Observations:   10               BIC:              13.2254 
Df Model:           1                Log-Likelihood:   -4.3101 
Df Residuals:       8                LL-Null:          -6.7301 
Converged:          1.0000           LLR p-value:      0.027807
No. Iterations:     6.0000           Scale:            1.0000  
-----------------------------------------------------------------
          Coef.    Std.Err.      z      P>|z|     [0.025   0.975]
-----------------------------------------------------------------
x1        0.6622     0.4001    1.6551   0.0979   -0.1220   1.4464
const    -3.6956     2.2889   -1.6145   0.1064   -8.1818   0.7906

